# GS01 — Hello Genesis: Load a Robot Scene

### Lab Description

This introductory lab builds a complete Genesis simulation from initialization to recorded output. You will create a scene, add a plane and a Franka Emika Panda arm, configure a camera, and advance the physics simulation in headless mode.

Genesis combines a physics simulator and visualizer inside each `Scene`. On AMD hardware, this course uses the `gs.amdgpu` backend so simulation kernels execute through ROCm.

#### Recommended Hardware

An AMD GPU supported by ROCm, such as an AMD Radeon™ GPU or AMD Ryzen™ AI processor with integrated Radeon graphics.

#### Software Environment

OS: Ubuntu 24.04 LTS  
Install [AUP Learning Cloud](https://amdresearch.github.io/aup-learning-cloud/installation/quick-start.html). The Genesis Simulation image provides ROCm, PyTorch, and `genesis-world==1.3.1`.

## Goals

- Initialize Genesis with the AMD GPU backend.
- Create and configure a headless Genesis scene.
- Add a plane, Franka Panda robot, and fixed camera.
- Understand the `build()` and `step()` simulation lifecycle.
- Render camera data and record a simulation video.

In [ ]:
# Suppress warning messages for clearer output
import os
import warnings

os.environ["TI_LOG_LEVEL"] = "error"
warnings.filterwarnings("ignore")
os.makedirs("Videos", exist_ok=True)

## Backend Initialization

Genesis supports multiple backends for parallel simulation. For AMD hardware, the recommended backend is ROCm (HIP).

In addition to choosing the backend, you can configure various parameters during initialization:

Example:

```python
gs.init(
    precision           = '32',
    logging_level       = None,
    backend             = gs.amdgpu,
    theme               = 'dark',
)
```

Here, we use the default init settings and set backend to amdgpu.

In [ ]:
import genesis as gs
import numpy as np

assert "scene" not in globals(), "Scene already exists. Restart the kernel before rerunning this lab."
gs.init(backend=gs.amdgpu, theme="light")

## Create a Scene

In Genesis, all entities—such as robots, objects, cameras, and sensors—exist within a **Scene**. A Scene serves as the core container of the simulation, encapsulating two main components:

* **Simulator**: Defines the physical world and handles all physics computations.
* **Visualizer**: Renders the scene and manages the graphical display.

A scene can be created either **with** or **without** a viewer:

* `show_viewer=True`: Launches a GUI viewer, useful for debugging or interactive visualization.
* `show_viewer=False`: Runs in headless mode, ideal for large-scale training or server-side execution.

In this lab, we set `show_viewer` to `False` and use the default `SimOptions` and `ViewerOptions` settings.
Later, instead of rendering the scene directly in this notebook, we will save the rendering output as a video for visualization.

Note that you can only create the scene ONCE. Recreating it will cause an ERROR. 


In [ ]:
scene = gs.Scene(show_viewer=False)

## Load a Robot

Once a scene has been created in Genesis, the next step is to populate it with robots, objects, or other physical entities. Genesis follows a fully **object-oriented design**, where every element in the simulation world is represented as an [**`Entity`**](https://genesis-world.readthedocs.io/en/latest/api_reference/entity/index.html).

An `Entity` is the abstraction for everything in the scene that requires physics simulation. This includes rigid or deformable bodies, terrains, fluids, and, of course, robots. Entities are introduced into a scene via the function `scene.add_entity`.

The first argument to `add_entity` is a **morph**, which encapsulates both the **geometry** and **pose** of an entity.

Different morph types allow you to load entities from:

* **Shape primitives** (e.g., planes, spheres, cubes)
* **Meshes**
* **URDF** files (Universal Robotics Description Format)
* **MJCF** files (MuJoCo's Robotics Format)
* **Terrains**
* **Soft robot descriptions**

This flexibility means that most commonly used robotics models—such as those described in URDF or MJCF—can be seamlessly integrated into Genesis.

In [ ]:
# Load Entity

plane = scene.add_entity(
    gs.morphs.Plane(),
)
franka = scene.add_entity(
    gs.morphs.MJCF(file="xml/franka_emika_panda/panda.xml"),
)

## Add a Camera

Next, we add a camera to record what happens in the scene.

In [ ]:
cam = scene.add_camera(
    res=(640, 480),
    pos=(3.5, 0.0, 2.5),
    lookat=(0, 0, 0.5),
    fov=30,
    GUI=True,
)

print("Successfully loaded a camera.")

## Build the Scene

After creating a scene, it must be built explicitly by calling `scene.build()`. This step is required because Genesis uses just-in-time (JIT) compilation to generate GPU kernels on the fly. Building the scene initializes this process, allocates device memory, and sets up the underlying data structures required for simulation.

In [ ]:
scene.build()

print("Successfully built the scene.")

## Start simulating

Then we start simulating and save it in the `Video` folder.

In [ ]:
# render rgb, depth, segmentation, normal
rgb, depth, segmentation, normal = cam.render(rgb=True, depth=True, segmentation=True, normal=True)
cam.start_recording(save_to_filename="Videos/video_01.mp4", fps=60)

for _ in range(100):
    scene.step()
    cam.render()

cam.stop_recording()

## Show the video

If everything works correctly, you will see a robotic arm appear on the screen and naturally fall due to gravity.

In [ ]:
from IPython.display import Video

Video(url="Videos/video_01.mp4")

## Conclusions

You initialized Genesis on an AMD GPU, built a headless scene, loaded a Franka Panda robot, rendered camera data, and recorded a simulation. In GS02, you will replace passive motion with explicit joint and PD control.

---

Copyright (C) 2026 Advanced Micro Devices, Inc. All rights reserved. Portions of this file consist of AI-generated content.  
SPDX-License-Identifier: MIT